In [2]:
#!/usr/bin/env python3
"""
Extract a balanced test set from aug_editing_plan.json.

This script samples data evenly across all safety principles.
"""

import json
import os
import random
import argparse
from collections import defaultdict
from typing import List, Dict, Any
import shutil

In [2]:
def extract_principle_id(safety_principle_text: str) -> int:
    """Extract principle ID from safety principle text."""
    if not safety_principle_text:
        return None
    try:
        # Extract the number before the first dot
        pid = int(safety_principle_text.strip().split('.')[0])
        return pid
    except (ValueError, IndexError):
        return None


def group_by_principle(data: List[Dict[str, Any]]) -> Dict[int, List[Dict[str, Any]]]:
    """Group data samples by their principle ID."""
    grouped = defaultdict(list)
    skipped = 0

    for item in data:
        principle = item.get('safety_risk', {}).get('safety_principle', '')
        pid = extract_principle_id(principle)

        if pid is None:
            skipped += 1
            continue

        grouped[pid].append(item)

    if skipped > 0:
        print(f"Warning: Skipped {skipped} samples with invalid principle format")

    return grouped


def extract_balanced_samples(
    grouped: Dict[int, List[Dict[str, Any]]],
    total_samples: int = 600,
    seed: int = 42
) -> List[Dict[str, Any]]:
    """
    Extract samples evenly distributed across principles.

    For each principle, we try to take `total_samples / num_principles` samples.
    If a principle has fewer samples than required, we take all available samples.
    """
    random.seed(seed)

    num_principles = len(grouped)
    samples_per_principle = total_samples // num_principles

    print(f"Target: {total_samples} samples across {num_principles} principles")
    print(f"Ideal samples per principle: {samples_per_principle}\n")

    selected_samples = []
    total_selected = 0

    for pid in sorted(grouped.keys()):
        available = grouped[pid]
        num_to_select = min(samples_per_principle, len(available))

        # Randomly sample from this principle
        sampled = random.sample(available, num_to_select)
        selected_samples.extend(sampled)

        print(f"Principle {pid:2d}: selected {num_to_select:3d} / {len(available):3d} available")

    total_selected = len(selected_samples)
    print(f"\nTotal selected: {total_selected} samples")

    if total_selected < total_samples:
        print(f"Note: Could only select {total_selected} samples (target was {total_samples})")
        print("      due to limited data in some principles.")

    return selected_samples

In [ ]:
input = 'data/metadata/safepair/safe_scenario_info.json' # 'data/metadata/unsafe_scenario_info.json'
output = 'data/metadata/safepair/test/test_list.json' # 'data/metadata/test/test_list.json'
total_samples = 300
seed = 42

# Load input data
print(f"Loading data from {input}...")
with open(input, 'r') as f:
    data = json.load(f)
print(f"Loaded {len(data)} samples\n")

# Group by principle
grouped = group_by_principle(data)

# Extract balanced samples
selected_samples = extract_balanced_samples(
    grouped,
    total_samples=total_samples,
    seed=seed
)

# Save output
os.makedirs(os.path.dirname(output), exist_ok=True)
with open(output, 'w') as f:
    json.dump(selected_samples, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(selected_samples)} samples to {output}")

Loading data from data/safepair/safe_scenario_info.json...
Loaded 6000 samples

Target: 300 samples across 33 principles
Ideal samples per principle: 9

Principle  1: selected   9 / 152 available
Principle  2: selected   9 / 343 available
Principle  3: selected   9 / 188 available
Principle  4: selected   9 /  65 available
Principle  5: selected   9 / 172 available
Principle  6: selected   9 / 227 available
Principle  7: selected   9 /  68 available
Principle  8: selected   9 /  27 available
Principle  9: selected   9 /  78 available
Principle 10: selected   9 / 104 available
Principle 11: selected   9 / 321 available
Principle 12: selected   9 / 355 available
Principle 13: selected   9 / 312 available
Principle 14: selected   9 / 262 available
Principle 15: selected   9 / 246 available
Principle 16: selected   9 / 164 available
Principle 17: selected   9 / 193 available
Principle 18: selected   9 / 177 available
Principle 19: selected   9 / 246 available
Principle 20: selected   9 / 2

In [4]:
# Copy edit images with original directory structure (DO NOT modify annotation paths)
annotation_file = 'data/safepair/test/test_list.json'

# Load annotation data (only to get the list of images to copy)
with open(annotation_file, 'r') as f:
    data = json.load(f)

# Copy images
copied = 0
skipped = 0
new_data = []

for i in range(len(data)):
    item = data[i]
    edit_image_path = item.get('safety_risk', {}).get('edit_image_path', '')
    if not edit_image_path:
        skipped += 1
        continue
    
    source_path = edit_image_path
    file_name = os.path.basename(source_path)
    dest_path = os.path.join('data/safepair/test/edit_image', file_name)
    
    # Handle annotate_image
    source_path_anno = source_path.replace('edit_image', 'annotate_image')
    dest_path_anno = dest_path.replace('edit_image', 'annotate_image')
    
    # Create directories if needed
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    os.makedirs(os.path.dirname(dest_path_anno), exist_ok=True)
    
    # Copy edit_image
    if os.path.exists(source_path) and not os.path.exists(dest_path):
        shutil.copy2(source_path, dest_path)
        copied += 1
        item['safety_risk']['edit_image_path'] = dest_path
        new_data.append(item)

        # Copy annotate_image if exists
        if os.path.exists(source_path_anno):
            shutil.copy2(source_path_anno, dest_path_anno)
    else:
        skipped += 1
        if not os.path.exists(source_path):
            print(f"not found: {source_path}")
        if os.path.exists(dest_path):
            print(f"Existing: {dest_path}")

print(f"\nCopied {copied} edit images")
print(f"Skipped {skipped} images")
print(f"\nNote: annotation_info.json paths were NOT modified")
print(len(new_data))

with open('data/safepair/test/test_list.json', 'w') as f:
    json.dump(new_data, f, indent=2, ensure_ascii=False)

Existing: data/safepair/test/edit_image/0000119.jpg
Existing: data/safepair/test/edit_image/0000073_2.jpg
Existing: data/safepair/test/edit_image/0000085.jpg
Existing: data/safepair/test/edit_image/0000230.jpg
Existing: data/safepair/test/edit_image/0000063_8.jpg
Existing: data/safepair/test/edit_image/0000085.jpg
Existing: data/safepair/test/edit_image/0000043.jpg

Copied 290 edit images
Skipped 7 images

Note: annotation_info.json paths were NOT modified
290


In [5]:
with open("data/safepair/safe_scenario_info.json") as f:
    data = json.load(f)
with open("data/safepair/test/test_list.json") as f:
    testset = json.load(f)

new_data = []
for d in data:
    flag = 0
    for d_test in testset:
        if d['image_path'] == d_test["image_path"]:
            flag = 1; continue
    if flag == 0 and d['safety_risk'] is not None and d['safety_risk']['editing_plan'] is not None:
        new_data.append(d)
print(len(data))
print(len(new_data))
with open("data/safepair/safe_training_list.json", "w") as f:
    json.dump(new_data, f, indent=2)

6000
5710


# Split dataset for SFT and RFT

In [25]:
with open("data/safepair/safe_training_list.json") as f:
    data_safe = json.load(f)
with open("data/unsafe_training_list.json") as f:
    data_unsafe = json.load(f)

print(data_safe[0])
random.shuffle(data_safe)
print(data_safe[0])
random.shuffle(data_unsafe)

ratio=0.2
split_idx = int(len(data_safe) * ratio)
data_safe_sft, data_safe_rft = data_safe[:split_idx], data_safe[split_idx:]

grouped = group_by_principle(data_unsafe)
selected_samples = extract_balanced_samples(
    grouped,
    total_samples=660,
    seed=seed
)

other_samples = []
for d in data_unsafe:
    flag = 0
    for sd in selected_samples:
        if d['image_path'] == sd['image_path']:
            flag = 1
            continue
    if flag == 0:
        other_samples.append(d)
print(len(data_unsafe))
print(len(other_samples))

split_idx = int(len(data_unsafe) * ratio) - 660
data_unsafe_sft, data_unsafe_rft = other_samples[:split_idx], other_samples[split_idx:]
data_unsafe_sft.extend(selected_samples)

sft_data = []
sft_data.extend(data_safe_sft)
sft_data.extend(data_unsafe_sft)
print(len(sft_data))
random.shuffle(sft_data)

rft_data = []
rft_data.extend(data_safe_rft)
rft_data.extend(data_unsafe_rft)
print(len(rft_data))
random.shuffle(rft_data)

with open('data/sft_training_list.json', 'w') as f:
    json.dump(sft_data, f, indent=2)

with open('data/rft_training_list.json', 'w') as f:
    json.dump(rft_data, f, indent=2)

{'image_path': 'data_ca/base_image/007125.png', 'scene_type': '6', 'safety_risk': {'action': 'Rinse the toothbrush under the faucet', 'editing_plan': 'No editing required', 'hazard_related_area': {'target_object': ['toothbrush', 'faucet'], 'constraint_object': []}, 'safety_principle': '5. Water & Electricity Separation', 'safety_hazard': None, 'pre_image_path': 'data_ca/base_image/007125.png', 'edit_image_path': 'data_ca/safepair/edit_image/007125.png', 'fidelity_check': 'ACCEPTED', 'hazard_check': 'ACCEPTED', 'annotation': {'target_object': {'toothbrush': {'bbox_2d': [539, 519, 570, 698], 'state': None, 'bbox_norm': [526, 675, 556, 908]}, 'faucet': {'bbox_2d': [593, 587, 686, 756], 'state': None, 'bbox_norm': [579, 764, 669, 984]}}, 'constraint_object': {}}, 'cot': '<think> Based on the action instruction "Rinse the toothbrush under the faucet", the target object to be interacted with is: [target_object][toothbrush][526, 675, 556, 908][]. Rinsing the toothbrush (a non-electrical objec